# Dealer gamma (GEX) regimes with FinzData

Point-in-time dealer gamma exposure from [FinzData](https://finzdata.com/point-in-time-gex): gamma flip, call wall, put wall and net gamma for 11,200+ US stocks, ETFs and indexes since 2010, each row stamped with the time it became known (17:00 New York).

A free key (email only, [finzdata.com/agents](https://finzdata.com/agents)) returns the last 30 days for every ticker. Full history for one ticker is $59, every ticker $295 ([shop](https://finzdata.com/shop)).

```
pip install finzdata
export FINZDATA_API_KEY=...
```

In [1]:
import finzdata as fz

g = fz.gex("SPX")  # last 30 days on a free key
g[["date", "spot", "flip", "call_wall", "put_wall", "net_gex_bn", "pcr_oi", "atm_iv_pct"]].tail(10)

         date         spot     flip  call_wall  put_wall  net_gex_bn  \
10 2026-10-21  7756.149110  7703.13     8000.0    7700.0   45.995219   
11 2026-10-22  7646.847310  7699.12     8000.0    7000.0  -34.309218   
12 2026-10-23  7625.291919  7685.16     8000.0    7000.0  -36.638993   
13 2026-10-26  7680.473840  7690.97     8000.0    7000.0   -9.643886   
14 2026-10-27  7736.155761  7685.09     8000.0    7700.0   40.210722   
15 2026-10-28  7738.857327  7687.77     8000.0    7700.0   42.113834   
16 2026-10-29  7705.671015  7705.37     8000.0    7700.0    1.024923   
17 2026-10-30  7559.664929  7669.22     8000.0    7000.0  -50.521012   
18 2026-11-03  7569.711642  7649.25     8000.0    7000.0  -40.418959   
19 2026-11-04  7688.620100  7666.70     8000.0    7000.0   25.154507   

      pcr_oi  atm_iv_pct  
10  1.599763    13.27525  
11  1.589126    18.37405  
12  1.549739    10.88340  
13  1.553255    12.11110  
14  1.559459    14.69290  
15  1.576942    16.44265  
16  1.566584    14

## Long or short gamma?

When the close is above the gamma flip, dealers are long gamma and tend to dampen moves (sell rallies, buy dips). Below the flip they are short gamma and tend to amplify moves.

In [2]:
g["regime"] = (g["spot"] >= g["flip"]).map({True: "long gamma", False: "short gamma"})
g["dist_to_flip_pct"] = (g["spot"] / g["flip"] - 1) * 100
g[["date", "spot", "flip", "dist_to_flip_pct", "regime"]].tail(10).round(2)

         date     spot     flip  dist_to_flip_pct       regime
10 2026-10-21  7756.15  7703.13              0.69   long gamma
11 2026-10-22  7646.85  7699.12             -0.68  short gamma
12 2026-10-23  7625.29  7685.16             -0.78  short gamma
13 2026-10-26  7680.47  7690.97             -0.14  short gamma
14 2026-10-27  7736.16  7685.09              0.66   long gamma
15 2026-10-28  7738.86  7687.77              0.66   long gamma
16 2026-10-29  7705.67  7705.37              0.00   long gamma
17 2026-10-30  7559.66  7669.22             -1.43  short gamma
18 2026-11-03  7569.71  7649.25             -1.04  short gamma
19 2026-11-04  7688.62  7666.70              0.29   long gamma

In [3]:
ax = g.set_index("date")[["spot", "flip", "call_wall", "put_wall"]].plot(figsize=(9, 4), title="SPX close vs gamma flip and walls")

## What the full history shows

Measured on SPX over 4,221 sessions (2010 to October 2026): next-day volatility was 24.5% annualised after net-short-gamma closes vs 13.5% after net-long; 72 of the 88 sessions that moved more than 3% followed a short-gamma close; SPX closed between its put and call walls on 74% of sessions.

Today's levels for 30 tickers: [finzdata.com/gex](https://finzdata.com/gex/). From an AI assistant: connect `https://api.finzdata.com/mcp` and ask *"Using FinzData, is SPX in positive or negative gamma today?"*